# Group chat findings

A summary of everything `eda.py` found in the group chat: who talks, when, how people like and react,
how they write, the chat's lore (names, nicknames, themes, polls), and the daily games.

**Before running:** run the first two pipeline steps from `bank_generation/` so `data/eda_out/stats.json` exists:

```bash
python tools/html_to_json.py data/ -o data/chat.json
python tools/eda.py data/chat.json -o data/eda_out
```

Run this notebook from `bank_generation/`. It only reads `data/eda_out/stats.json`.

> **Privacy:** once run, this notebook's outputs contain real chat content. Clear the outputs before committing it
> (*Edit → Clear All Outputs*, or `jupyter nbconvert --clear-output --inplace findings.ipynb`).

In [ ]:
import json, re, warnings
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", message="Glyph")   # emoji / non-Latin names the default font can't draw

STATS = Path("data/eda_out/stats.json")
S = json.load(open(STATS, encoding="utf-8"))
M = S["meta"]

BLUE, INK2, GRID = "#2a78d6", "#52514e", "#e4e3df"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "axes.axisbelow": True})

def show(text):
    display(Markdown(text))

def cell(v):
    if isinstance(v, int): return f"{v:,}"
    if isinstance(v, list): return ", ".join(map(str, v))
    return str(v).replace("|", "\\|").replace("\n", " ")

def table(rows, headers, n=None):
    rows = list(rows)[:n] if n else list(rows)
    lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    lines += ["| " + " | ".join(cell(c) for c in r) + " |" for r in rows]
    show("\n".join(lines))

def label(name):
    """Chart-safe name: drop characters the default font can't draw."""
    return re.sub(r"[^\u0000-\u024F]", "", str(name)).strip() or str(name)

def barh(pairs, title, xlabel="", n=16):
    pairs = list(pairs)[:n][::-1]
    fig, ax = plt.subplots(figsize=(7, 0.32 * len(pairs) + 1))
    ax.barh([label(p[0]) for p in pairs], [p[1] for p in pairs], color=BLUE)
    ax.set_title(title, loc="left"); ax.set_xlabel(xlabel); ax.grid(axis="y", visible=False)
    plt.tight_layout(); plt.show()

def d(ts):
    return datetime.fromisoformat(ts)

## Headlines

In [ ]:
first, last = d(M["first_ts"]), d(M["last_ts"])
span_days = (last.date() - first.date()).days + 1
total_sent = sum(n for _, n in S["messages_per_person"])
top_p, top_n = S["messages_per_person"][0]
peak_month = max(S["per_month"], key=lambda x: x[1])
busy_day = S["busiest_days"][0]
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
wd, hr = max(((w, h) for w in range(7) for h in range(24)), key=lambda x: S["weekday_hour"][x[0]][x[1]])
pair = S["top_like_pairs"][0]
longest_name = S["group_names_total"][0]
most_nicked = max(S["nicknames_by_person"].items(), key=lambda kv: len(kv[1]))
laugh = S["laugh_totals"][0]

show(f"""
- **{M['title']}**: {M['message_count']:,} entries from {first:%b %d, %Y} to {last:%b %d, %Y}, active on {S['active_days']} of {span_days} days.
- **{len(S['messages_per_person'])} senders.** {top_p} sent the most: {top_n:,} messages ({top_n / total_sent:.0%} of all).
- **Busiest month:** {peak_month[0]} ({peak_month[1]:,} messages). **Busiest day:** {busy_day[0]} ({busy_day[1]:,}).
- **Peak hour:** {days[wd]} around {hr:02d}:00.
- **{S['likes_total']:,} likes** and **{S['reactions_total']:,} other reactions.** Biggest fan: {pair[0]} liked {pair[1]}'s messages {pair[2]:,} times.
- **Funniest** (most laugh reactions received): {S['funniest_by_laugh_reactions'][0][0]}. Most common laugh: {laugh[0]} ({laugh[1]:,} times).
- **{len(S['group_names'])} group renames.** Longest-lived name: "{longest_name[0]}" ({longest_name[1]} days). Top renamer: {S['group_renamers'][0][0]}.
- **{len(S['nicknames'])} nickname changes.** Most nicknamed: {most_nicked[0]} ({len(most_nicked[1])} nicknames).
""")

## 1. Who talks

In [ ]:
barh(S["messages_per_person"], "Messages sent", "messages", n=24)
table(S["types"], ["entry type", "count"])

In [ ]:
seen = sorted(S["first_last_seen"].items(), key=lambda kv: kv[1][0])
table([(p, a[:10], b[:10]) for p, (a, b) in seen], ["person", "first message", "last message"])

### Writing style (regulars only)

In [ ]:
table(S["avg_words_per_message"], ["person", "avg words / message"])
table(S["all_caps_messages"], ["person", "ALL-CAPS messages", "% of theirs"])
table(S["questions_asked"], ["person", "questions", "% of theirs"])
table(S["longest_solo_runs"], ["person", "longest run of messages in a row"])

## 2. When the chat is active

In [ ]:
months = [m for m, _ in S["per_month"]]
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(months, [n for _, n in S["per_month"]], color=BLUE, marker="o", ms=3)
ax.set_title("Messages per month", loc="left")
ax.set_xticks(months[::3]); ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 3))
im = ax.imshow(S["weekday_hour"], aspect="auto", cmap="Blues")
ax.set_yticks(range(7)); ax.set_yticklabels(days)
ax.set_xticks(range(0, 24, 2)); ax.set_xlabel("hour of day"); ax.grid(False)
ax.set_title("Messages by weekday and hour", loc="left"); fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
table(S["busiest_days"], ["day", "messages"], n=10)
table(S["night_owls_1am_5am"], ["night owl (1–5 am)", "messages", "% of theirs"], n=8)
table(S["early_birds_5am_9am"], ["early bird (5–9 am)", "messages", "% of theirs"], n=8)
table(S["chat_revivers_after_6h"], ["person", "broke a 6h+ silence"], n=10)

## 3. Likes and reactions

In [ ]:
show(f"Like rules: ❤ from anyone, plus personal like emojis: "
     + ", ".join(f"{p} {' '.join(e)}" for p, e in M["like_rules"]["personal"].items()))
table(S["like_emoji_counts"], ["like emoji", "count"])
barh([(p, n) for p, n, _ in S["likes_given"]], "Likes given", "likes")
barh([(p, n) for p, n, _ in S["likes_received"]], "Likes received", "likes")

In [ ]:
table(S["likes_given"], ["person", "likes given", "% of others' messages liked"], n=16)
table(S["likes_received"], ["person", "likes received", "likes per message"], n=16)
table(S["top_like_pairs"], ["fan", "liked", "times"], n=15)

In [ ]:
table(S["reaction_emojis"], ["reaction", "count"], n=15)
table(S["funniest_by_laugh_reactions"], ["person", "laugh reactions received"], n=10)
table([(p, ", ".join(f"{e} {n}" for e, n in v[:6])) for p, v in S["reaction_emojis_by_person"].items()],
      ["person", "reactions they use most"])

### Most-reacted messages

In [ ]:
rows = [(m["ts"][:10], m["sender"], m["type"], (m["text"] or "")[:80], m["likes"], m["reactions"])
        for m in S["most_reacted_messages"][:15]]
table(rows, ["date", "sender", "type", "text", "likes", "reactions"])

In [ ]:
for p, msgs in S["most_liked_text_by_person"].items():
    if msgs:
        text, n = msgs[0]
        show(f"**{p}** ({n} likes): {text[:160]}")

## 4. How people write

In [ ]:
table(S["top_words"], ["word", "count"], n=25)
table(S["laugh_totals"], ["laugh", "count"])
table([(p, ", ".join(f"{e} {n}" for e, n in v[:4])) for p, v in S["laugh_styles"].items()], ["person", "how they laugh"])

In [ ]:
table([(p, ", ".join(w for w, *_ in v[:8])) for p, v in S["distinctive_words"].items()], ["person", "distinctive words"])
table([(p, ", ".join(f"{w} ({n})" for w, n in v[:5])) for p, v in S["top_phrases_by_person"].items()], ["person", "top phrases"])
table([(p, " ".join(e for e, _ in v[:8])) for p, v in S["text_emoji_by_person"].items()], ["person", "favourite emoji"])

In [ ]:
table(S["name_mentions"], ["person", "times their name comes up"], n=16)
table(S["at_mentions"], ["person", "@-mentions"], n=10)

## 5. Chat lore

### Group names

In [ ]:
show(f"Current name: **{S['current_title']}**")
table([(g["from"][:10], g["name"], g["set_by"], g["days"]) for g in S["group_names"]], ["from", "name", "set by", "days"])
barh([(n, days) for n, days, *_ in S["group_names_total"]], "Days each name was in use", "days", n=15)
table(S["group_renamers"], ["person", "renames"])

### Nicknames

In [ ]:
for p, nicks in sorted(S["nicknames_by_person"].items(), key=lambda kv: -len(kv[1])):
    unique = list(dict.fromkeys(nicks))
    show(f"**{p}** ({len(unique)}): " + "; ".join(unique))

### Themes, polls, calls and membership

In [ ]:
table(Counter(t for _, _, t in S["themes"]).most_common(10), ["theme", "times set"])
table(Counter(p for _, p, _ in S["themes"]).most_common(), ["person", "themes set"])
table(S["polls"], ["date", "created by", "poll"])
table(S["poll_votes"], ["option", "votes"], n=10)
table(S["calls_started"], ["person", "calls started"])
table(S["membership"], ["date", "event"])

## 6. Media and shares

In [ ]:
for kind, rows in S["media_by_person"].items():
    show(f"**{kind}**: " + ", ".join(f"{p} {n:,}" for p, n in rows[:6]))
table(S["top_shared_accounts"], ["account", "posts shared"], n=15)
table(S["top_link_domains"], ["domain", "links"], n=10)

## 7. Daily games

In [ ]:
table([(g, sum(n for _, n in v), ", ".join(f"{p} {n}" for p, n in v[:5])) for g, v in S["games_posted"].items()],
      ["game", "results posted", "top posters"])
table(S["krillion_scores"], ["person", "Krillion dives posted", "avg score", "best"])
table(S["wordle_avg"], ["person", "Wordles posted", "avg guesses (X = 7)"])